# Step 3 — Settling fit for one or many runs (SettlingFit v01)

For each run in a list, fits the settling of the particle population to the OBS decay using the **calibration file** from `2_SettlingCalibration_v01.ipynb` (step 2). Nothing is calibrated here.

**Inputs per run:** the OBS decay csv (the `*_time_series_decayONLY.csv` file from step 1: `sec, NTU, Co_NTU_avg, Cbackground_NTU_avg`) and the initial particle size distribution csv. Tank geometry, rotation rate, $K_0$, $K_{min}$ and the ADV decay shape come from the calibration file.

**What is fitted** (`what_to_fit`, can differ per run):

| value | fitted unknown | class velocities |
|---|---|---|
| `'rhos'` | particle density (solid particles, $n_f$ fixed) | settling law at each class diameter |
| `'nf'` | fractal dimension of flocs (density and primary size fixed) | settling law at each class diameter |
| `'ws50'` | settling velocity at the median size, no density or $n_f$ assumption | $w_i=w_{50}(d_i/d_{50})^\beta$ |

**Outputs:** a results table (also saved as csv) with the fitted value and its 1-σ, the velocity at the median size and its range, the mean velocity, the fit rms, and the fitted value if $K_0$ is divided or multiplied by `K0_factor` (the effect of the turbulence uncertainty); and one data/model/residual plot per run. A single size is a PSD file with one row.

In [ ]:
# --- Find the project folder, then work inside its work/ folder ----------------
# All data paths below (data_OBS/, calibrations/, results/, ...) are relative to work/.
import os
import sys
_root = os.getcwd()
for _ in range(4):   # this folder, then up to three levels above
    if os.path.isdir(os.path.join(_root, 'tools')) and os.path.isdir(os.path.join(_root, 'work_template')):
        break
    _root = os.path.dirname(_root)
else:
    raise FileNotFoundError('could not find the project folder (the one containing tools/ and work_template/)')
sys.path.insert(0, _root)                          # for "from tools ..." imports
sys.path.insert(0, os.path.join(_root, 'tools'))   # for "import settling_tools"
if not os.path.isdir(os.path.join(_root, 'work')):
    from setup_work import setup_work
    setup_work(_root)                              # first run: create work/ from work_template/
os.chdir(os.path.join(_root, 'work'))
print('working in:', os.getcwd())
# ---------------------------------------------------------------------------------

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import settling_tools as ss

%config InlineBackend.figure_format='retina'

## 1. User input

In [ ]:
# --- User input -----------------------------------------------------------
calibration_file = 'calibrations/cal_60rpm.json'     # from 2_SettlingCalibration_v01.ipynb (same mixer rotation rate as the runs below)
rpm = 60                                              # rotation rate of the runs below (checked against the calibration); None to skip the check

# the runs: one dict each. Required: name, obs_file, psd_file. Any default below can be overridden per run
# (e.g. T_C, S_ppt, coef, nf, dp_um, rhos_for_nf, beta, gain_exp, what_to_fit, t_fit_start, t_fit_end, psd).
runs = [
    dict(name='standard 75-90 um (PSD file)', obs_file='data_OBS/MicroSphere_75_90_SAMPLE_time_series_decayONLY.csv',
         psd_file='data_PSD/PSD_example_75_90.csv'),
    dict(name='standard, single size 82.5 um', obs_file='data_OBS/MicroSphere_75_90_SAMPLE_time_series_decayONLY.csv',
         psd_file='data_PSD/PSD_example_single_82_5um.csv'),
]

# defaults for all runs --------------------------------------------------
what_to_fit = 'rhos'      # 'rhos', 'nf' or 'ws50'
T_C, S_ppt  = 19.4, 8.0   # water temperature [deg C] and salinity [ppt]
coef  = 3                 # settling law: 1 flocs, 2 viscous flocs, 3 sieve sand / smooth spheres, 4 natural sand, 5 sphere
nf    = 3.0               # fractal dimension used when what_to_fit = 'rhos' (3 = solid particle)
dp_um = 8.0               # primary particle size [um] (no effect when nf = 3)
rhos_for_nf = 2650.0      # density used when what_to_fit = 'nf'
beta  = 1.3               # d-exponent of the velocity spectrum for 'ws50' (about 2 in the Stokes range, about 1.3 for 75-90 um sand-like spheres)
gain_exp = 0.0            # OBS response per unit amount scales as d^-gain_exp (0 = PSD amounts are signal fractions)
psd = dict(d_col=None, p_col=None, p_type='fraction', d_units='um', d_min=None, d_max=None)   # PSD csv layout (see settling_tools.load_psd)
n_max_classes = 12        # a finer PSD is merged into at most this many classes
t_fit_start, t_fit_end = 0.0, None    # fit window [s since mixer off]

do_sensitivity = True     # also refit with K0 / K0_factor and K0 * K0_factor (about 3 x longer)
K0_factor = 1.4           # K0 uncertainty factor (see the K0 vs standard density table in the calibration notebook)

# geometry: from the calibration file by default; set to a Geometry to override (e.g. ss.Geometry(h=0.1778, z_obs=0.0635))
geometry_override = None

# output
results_csv = 'SettlingFit_results.csv'
plot_dir    = 'fit_plots'
# --------------------------------------------------------------------------

# Make the output folder
results_folder = 'results/'
if os.path.isdir(results_folder) != 1:
    os.mkdir(results_folder)

## 2. Calibration

In [ ]:
cal = ss.load_calibration(calibration_file)

# Tank geometry: from the calibration file unless overridden in the input cell
geom = geometry_override or cal['geom']

print(f"calibration {calibration_file}: {cal['turb_mode']} model, "
      f"K0 = {cal['K0']:.3g} m2/s, Kmin = {cal['Kmin']:.3g} m2/s, {cal['rpm']} rpm, created {cal['created']}")
print(f"geometry: h = {geom.h:.4f} m, sensor {geom.z_obs:.4f} m above the bottom ({geom.z_s:.4f} m below the surface)")

# If the calibration notebook tested other densities for the standard, show the resulting K0 range
if cal.get('K0_vs_standard_density'):
    sd = pd.DataFrame(cal['K0_vs_standard_density'])
    print(f"K0 range for the standard's density {sd.rho_standard.min():.0f}-{sd.rho_standard.max():.0f} kg/m3: "
          f"{sd.K0.min():.2e} - {sd.K0.max():.2e} m2/s")

# Warn if the geometry or rotation rate differs from the calibration
for warning in ss.check_calibration(cal, geom, rpm):
    print('WARNING:', warning)

# --- Plot the K(t) used for the fits, over the first 60 s ---------------------
tp = np.linspace(0, 60, 300)
if cal['shape'] is not None:
    # ADV model: K(t) = Kmin + K0*shape(t)
    K_of_t = ss.shape_K(cal['shape'], cal['K0'], cal['Kmin'])
    Kt = np.array([K_of_t(x) for x in tp])
else:
    # exponential model: K(t) = Kmin + K0*exp(-t/tau)
    Kt = cal['Kmin'] + cal['K0'] * np.exp(-tp / cal['tau_used'])

plt.figure(figsize=(6, 3))
plt.semilogy(tp, Kt)
plt.xlabel('Time since mixer off [s]')
plt.ylabel('K [m$^2$/s]')
plt.title('K(t) used')
plt.tight_layout()

## 3. Fit all runs

In [ ]:
# Defaults from the input cell; a run's own dict can override any of them
defaults = dict(what_to_fit=what_to_fit, T_C=T_C, S_ppt=S_ppt, coef=coef, nf=nf, dp_um=dp_um,
                rhos_for_nf=rhos_for_nf, beta=beta, gain_exp=gain_exp, psd=psd,
                t_fit_start=t_fit_start, t_fit_end=t_fit_end, K0_factor=K0_factor)

os.makedirs(results_folder+plot_dir, exist_ok=True)
rows = []   # one results row per run (becomes the results table)
fits = {}   # fit and data per run name, for the class-velocity table below

for run in runs:
    o = {**defaults, **run}   # this run's settings: defaults, overridden by the run's own values

    # --- Fit this run --------------------------------------------------------
    row, fit, dat = ss.fit_run(o['obs_file'], o['psd_file'], geom, cal, o['T_C'], o['S_ppt'], o['coef'],
                               what=o['what_to_fit'], nf=o['nf'], dp_um=o['dp_um'],
                               rhos_for_nf=o['rhos_for_nf'], beta=o['beta'], gain_exp=o['gain_exp'],
                               psd_kw=o['psd'], n_max_classes=n_max_classes,
                               t_start=o['t_fit_start'], t_end=o['t_fit_end'],
                               K0_factor=o['K0_factor'], do_sensitivity=do_sensitivity, name=o['name'])
    row['psd_file'] = os.path.basename(o['psd_file'])
    row['obs_file'] = os.path.basename(o['obs_file'])
    rows.append(row)
    fits[o['name']] = (fit, dat)
    print(f"{o['name']}: {o['what_to_fit']} = {row['value']:.4g} +/- {row['sigma']:.2g} {row['units']}   "
          f"w50 = {row['w50_mm_s']:.3f} mm/s   rms = {row['rms']:.3f}")

    # --- Model curves ----------------------------------------------------------
    t, c = dat['t'], dat['c']
    t_plot = np.linspace(0, t.max(), 400)   # smooth model curve for plotting
    c_fit = ss.model_c_psd(t_plot, fit, geom, cal['K0'], cal['tau_used'], cal['Kmin'], shape=cal['shape'])
    c_at = ss.model_c_psd(t, fit, geom, cal['K0'], cal['tau_used'], cal['Kmin'], shape=cal['shape'])  # at the data times, for residuals

    # --- Plot: data vs model (top) and residuals (bottom) ----------------------
    fig, ax = plt.subplots(2, 1, figsize=(7.5, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[3, 1.3]))

    ax[0].plot(t, c, '.', ms=3, color='0.6', label='OBS')
    ax[0].plot(t_plot, c_fit, lw=2, label=f"model, w50 = {row['w50_mm_s']:.3f} mm/s")
    ax[0].set_ylabel('c  [-]')
    ax[0].set_title(o['name'])
    ax[0].legend()

    ax[1].plot(t, c - c_at, '.', ms=3)
    ax[1].axhline(0, color='k', lw=0.8)
    ax[1].set_xlabel('Time since mixer off [s]')
    ax[1].set_ylabel('residual')

    plt.tight_layout()

    # Save the plot, named after the run (characters other than letters, digits, '-' and '_' become '_')
    safe_name = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in o['name'])
    fig.savefig(os.path.join(results_folder+plot_dir, safe_name + '.png'), dpi=150)
    plt.show()

## 4. Results

In [ ]:
res = pd.DataFrame(rows)

# save the results
res.to_csv(results_folder+results_csv, index=False)
print(f'saved {results_csv}  ({len(res)} runs)')

# Columns to show (only those present; the K0_low/high columns exist only with do_sensitivity)
wanted = ('run', 'what', 'value', 'sigma', 'units', 'value_K0_low', 'value_K0_high',
          'w50_mm_s', 'w50_lo_mm_s', 'w50_hi_mm_s', 'd50_um', 'rms')
show = [col for col in wanted if col in res.columns]

# Numeric columns to 4 significant figures; text columns as they are
col_formats = {col: '{:.4g}' for col in show if col not in ('run', 'what', 'units')}
display(res[show].style.format(col_formats).hide(axis='index'))

## 5. Class velocities of one run

In [ ]:
name = runs[0]['name']   # change to inspect another run
fit, dat = fits[name]

# Diameter, signal fraction and fitted settling velocity of each size class
classes = pd.DataFrame({'d [um]': dat['d_um'], 'fraction': dat['p'], 'w [mm/s]': fit['ws'] * 1e3})
display(classes.round(3).style.hide(axis='index'))

## Notes

- `value_K0_low` / `value_K0_high` are the fitted values with $K_0$ divided / multiplied by `K0_factor`; together with `sigma` (a fit uncertainty only) they bracket the effect of the turbulence calibration. The K0 spread from the calibration notebook's density table is the guide for choosing `K0_factor`.
- Runs must share the mixer rotation rate, tank, water depth and sensor position of the calibration. Water temperature and salinity can differ per run.
- The PSD amounts are taken as the fraction of the initial OBS signal in each class (`gain_exp = 0`). For flocs the PSD should be the floc size distribution at the time of the run.
- `'ws50'` is the most assumption-free summary of the settling; density and $n_f$ add the settling law on top.
- Results are written to `results_csv` and plots to `plot_dir`, both inside `results_folder`.